# 🤖 Multi-Agent AI Communication Framework for Request Handling

A simple, beginner-friendly Agentic AI project that demonstrates **four communication frameworks**
between AI agents:

1. **Centralized** – a Coordinator controls everything
2. **Decentralized** – agents pass work along a fixed chain, no coordinator
3. **Parallel** – agents run concurrently using `asyncio`
4. **Peer-to-Peer** – agents talk directly to each other, with one optional correction cycle

Run each cell in order (top to bottom) in Google Colab.


In [ ]:
# CELL 1: Install required packages
# gradio -> for the UI
# google-genai -> to call the Gemini API
!pip install -q gradio google-genai


In [ ]:
# CELL 2: Import libraries and configure the Gemini API
import os
import time
import asyncio
import traceback
from getpass import getpass

from google import genai
from google.genai import types

# --------------------------------------------------------------------
# API KEY SETUP
# --------------------------------------------------------------------
# Option A (recommended for Colab): run this cell and paste your key
# securely. It will NOT be printed or stored in the notebook file.
#
# You can get a FREE Gemini API key from Google AI Studio:
# https://aistudio.google.com/app/apikey
#
# Option B: if you already have GEMINI_API_KEY set as an environment
# variable (e.g. in a Colab secret), this cell will simply reuse it.
# --------------------------------------------------------------------

if not os.getenv("GEMINI_API_KEY"):
    try:
        os.environ["GEMINI_API_KEY"] = getpass("Enter your Gemini API Key: ")
    except Exception:
        # In non-interactive environments, just skip. The UI will show
        # a clear error later if the key is still missing.
        pass

API_KEY = os.getenv("GEMINI_API_KEY")

client = None
if API_KEY:
    client = genai.Client(api_key=API_KEY)
    print("✅ API key loaded. Client ready.")
else:
    print("⚠️  No API key found. The app will still work — it will "
          "automatically run in DEMO MODE (simulated responses) instead.")

# Model used for all agent calls. "gemini-flash-latest" is an alias that
# always points at Google's current recommended fast/cheap model, so this
# keeps working correctly even as specific dated model versions are
# retired over time. Change it to a specific model name if you prefer.
MODEL_NAME = "gemini-flash-latest"

# --------------------------------------------------------------------
# DEMO MODE
# --------------------------------------------------------------------
# If the Gemini API is unavailable for any reason (no key, no quota,
# invalid key, rate limit, network error), the app automatically falls
# back to a local simulated response generator so the project always
# stays demonstrable. Once we detect the API is unavailable, we flip
# USE_DEMO_MODE to True and stop calling the API again for the rest of
# the session (no repeated failing requests).
# --------------------------------------------------------------------
USE_DEMO_MODE = client is None
DEMO_MODE_REASON = "no API key configured" if USE_DEMO_MODE else ""


Enter your Gemini API Key: ··········
✅ API key loaded. Client ready.


In [ ]:
# CELL 3: Message class
# A Message represents a single piece of communication between two agents
# (or between the "User"/"Coordinator" and an agent).

class Message:
    def __init__(self, sender, receiver, content):
        self.sender = sender
        self.receiver = receiver
        self.content = content
        self.timestamp = time.time()

    def __repr__(self):
        return f"{self.sender} -> {self.receiver}: {self.content[:60]}"


In [ ]:
# CELL 4: MessageBus class
# Stores every communication that happens between agents so we can display
# a full communication log in the UI.

class MessageBus:
    def __init__(self):
        self.messages = []

    def send(self, sender, receiver, content):
        """Record a message being sent from one agent/role to another."""
        msg = Message(sender, receiver, content)
        self.messages.append(msg)
        return msg

    def clear(self):
        self.messages = []

    def get_log_text(self):
        """Return the full communication log as readable text for the UI."""
        if not self.messages:
            return "No messages yet."
        lines = []
        for i, m in enumerate(self.messages, start=1):
            lines.append(f"[{i}] {m.sender} → {m.receiver}\n{m.content}\n")
        return "\n".join(lines)

    def count(self):
        return len(self.messages)


In [ ]:
# CELL 5: Agent class and LLM calling function (with DEMO MODE fallback)
# Every agent shares the same basic structure: a name, a system prompt
# (its "role"/personality), and a run() method that calls the LLM.

# Counters so the UI can report "Number of LLM Calls" and demo usage.
llm_call_counter = {"count": 0}    # real Gemini API calls made
demo_call_counter = {"count": 0}   # simulated (DEMO MODE) calls made

DEMO_LABEL = "⚠️ DEMO MODE — Gemini API unavailable. Response is simulated for demonstration.\n\n"


def classify_gemini_error(e):
    """Convert Gemini exception into a useful human-readable reason."""

    text = str(e).lower()
    error_type = type(e).__name__.lower()

    # -----------------------------
    # 429 - Rate limit / quota
    # -----------------------------
    if (
        "resource_exhausted" in text
        or "quota exceeded" in text
        or "quota_exceeded" in text
        or "rate_limit_exceeded" in text
        or "too_many_requests" in text
        or "429" in text
    ):
        return "quota exceeded / rate limit reached"

    # -----------------------------
    # 401 - Authentication
    # -----------------------------
    if (
        "unauthenticated" in text
        or "401" in text
        or "authentication" in text
    ):
        return "invalid or missing API key"

    # -----------------------------
    # 403 - Permission
    # -----------------------------
    if (
        "permission_denied" in text
        or "403" in text
        or "api key not valid" in text
    ):
        return "invalid API key / permission denied"

    # -----------------------------
    # 404 - Model/resource
    # -----------------------------
    if (
        "not_found" in text
        or "404" in text
        or "model_not_found" in text
    ):
        return "model not found or unavailable"

    # -----------------------------
    # 400 - Bad request
    # -----------------------------
    if (
        "invalid_argument" in text
        or "400" in text
    ):
        return "invalid API request"

    # -----------------------------
    # 503 - Gemini temporarily unavailable
    # -----------------------------
    if (
        "503" in text
        or "service_unavailable" in text
        or "temporarily unavailable" in text
    ):
        return "Gemini service temporarily unavailable"

    # -----------------------------
    # 504 - Timeout
    # -----------------------------
    if (
        "deadline_exceeded" in text
        or "504" in text
        or "timeout" in text
        or "timed out" in text
    ):
        return "request timed out"

    # -----------------------------
    # Actual network/connection problem
    # -----------------------------
    if (
        "connectionerror" in error_type
        or "connecterror" in error_type
        or "connection refused" in text
        or "connection reset" in text
        or "network is unreachable" in text
        or "failed to establish a new connection" in text
    ):
        return "network connection error"

    # -----------------------------
    # Unknown
    # -----------------------------
    return f"API error: {type(e).__name__}"


def _split_request_and_context(user_content):
    """
    Our framework functions build user_content either as the raw request
    (first agent in a chain) or as "User request: ...\n\n<context>" (later
    agents). This pulls both parts back out so the demo generator can
    reference the actual request and the previous agent's output.
    """
    if "User request:" in user_content:
        after = user_content.split("User request:", 1)[1]
        parts = after.split("\n\n", 1)
        request_text = parts[0].strip()
        context_text = parts[1].strip() if len(parts) > 1 else ""
    else:
        request_text = user_content.strip()
        context_text = ""
    return request_text, context_text


def generate_demo_response(system_prompt, user_content):
    """
    Local, deterministic fallback used when the Gemini API is unavailable.
    It is NOT a fixed sentence — it builds a response from the agent's
    role, the actual user request, and (when available) the previous
    agent's output, so it still demonstrates the multi-agent flow.
    """
    request_text, context_text = _split_request_and_context(user_content)
    topic = request_text[:90] + ("..." if len(request_text) > 90 else "")
    prev_snippet = context_text[:120] + ("..." if len(context_text) > 120 else "") if context_text else ""
    role = system_prompt.lower()

    if "research" in role:
        body = (
            f"Simulated research on: \"{topic}\"\n"
            f"- Relevant background factors and typical real-world considerations for this topic.\n"
            f"- Common benefits usually associated with this kind of request.\n"
            f"- Common challenges or constraints usually associated with this kind of request.\n"
            f"- A rough cost/resource consideration to keep in mind."
        )
    elif "analysis" in role:
        source = f'based on the research notes: "{prev_snippet}"' if prev_snippet else f'based on the request: "{topic}"'
        body = (
            f"Simulated analysis {source}\n"
            f"- Advantage: this direction appears to address a real, stated need.\n"
            f"- Disadvantage: it likely requires upfront investment (time, cost, or training).\n"
            f"- Key factor: feasibility depends on available resources and stakeholder buy-in.\n"
            f"- Conclusion: worth pursuing with a phased/pilot approach."
        )
    elif "critic" in role:
        source = f'reviewing: "{prev_snippet}"' if prev_snippet else f'reviewing the request: "{topic}"'
        body = (
            f"Simulated critique {source}\n"
            f"- Missing: a concrete cost estimate is not specified.\n"
            f"- Weakness: risks and edge cases could be explored further.\n"
            f"- Suggestion: include a pilot phase before full-scale rollout.\n"
            f"- Overall: the reasoning direction looks sound but could use more supporting data."
        )
    else:  # response agent / fallback
        source = f'combining the available notes: "{prev_snippet}"' if prev_snippet else f'for the request: "{topic}"'
        body = (
            f"Simulated final answer, {source}\n"
            f"Overall, the request appears reasonable and achievable. Key points: it offers clear "
            f"benefits, comes with manageable challenges, and would benefit from a phased rollout "
            f"with cost tracking and a short pilot before full deployment."
        )

    return DEMO_LABEL + body


def call_llm(system_prompt, user_content):
    """
    Calls the LLM with a system prompt (agent role) and user content
    (the task/information it needs to work on). Returns the text reply.

    MODE 1 — Gemini available: calls the real API.
    MODE 2 — Gemini unavailable (no key / no quota / rate limit / network
    error): automatically falls back to generate_demo_response() and sets
    USE_DEMO_MODE so we stop retrying the failing API for the rest of the
    session. The app never crashes and never shows a raw traceback here.
    """
    global USE_DEMO_MODE, DEMO_MODE_REASON

    if USE_DEMO_MODE or client is None:
        demo_call_counter["count"] += 1
        return generate_demo_response(system_prompt, user_content)

    try:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=user_content,
            config=types.GenerateContentConfig(
                system_instruction=system_prompt,
                temperature=0.4,
                max_output_tokens=350,
            ),
        )
        llm_call_counter["count"] += 1
        text = (response.text or "").strip()
        if not text:
            # Gemini can return an empty response (e.g. blocked by a
            # safety filter) without raising an exception. Treat this as
            # a normal recoverable failure for this one call, not a
            # reason to switch the whole app into DEMO MODE.
            return "[ERROR] Empty response from Gemini (it may have been filtered)."
        return text
    except Exception as e:
        DEMO_MODE_REASON = classify_gemini_error(e)
        USE_DEMO_MODE = True
        # Concise diagnostic for the notebook console only — no traceback,
        # and the API key itself is never included/printed.
        print(f"[INFO] Gemini API unavailable ({DEMO_MODE_REASON}). Switching to DEMO MODE.")
        demo_call_counter["count"] += 1
        return generate_demo_response(system_prompt, user_content)


class Agent:
    """A simple, reusable agent structure used by all four agents."""

    def __init__(self, name, system_prompt):
        self.name = name
        self.system_prompt = system_prompt
        self.status = "Waiting"   # Waiting / Running / Completed / Failed

    def run(self, task_text):
        """Run this agent on a given piece of text and return its output."""
        self.status = "Running"
        try:
            output = call_llm(self.system_prompt, task_text)
            if output.startswith("[ERROR]"):
                self.status = "Failed"
            else:
                self.status = "Completed"
            return output
        except Exception:
            self.status = "Failed"
            return f"[ERROR] {traceback.format_exc(limit=1)}"

    async def run_async(self, task_text):
        """Async wrapper so this agent can run concurrently with others."""
        loop = asyncio.get_event_loop()
        return await loop.run_in_executor(None, self.run, task_text)


In [ ]:
# CELL 6: Create the four specialized agents, each with its own system prompt

RESEARCH_PROMPT = (
    "You are a research-oriented agent. Analyze the user's request and "
    "provide relevant information and considerations."
)

ANALYSIS_PROMPT = (
    "You are an analysis agent. Analyze the request and research "
    "information. Identify important factors, advantages, disadvantages, "
    "and conclusions."
)

CRITIC_PROMPT = (
    "You are a critic agent. Review the information and identify errors, "
    "missing information, contradictions, and weaknesses."
)

RESPONSE_PROMPT = (
    "You are the final response agent. Combine the available information "
    "and provide a clear final answer."
)

def make_fresh_agents():
    """Create brand-new agent instances (fresh 'Waiting' status) for a run."""
    return {
        "Research Agent": Agent("Research Agent", RESEARCH_PROMPT),
        "Analysis Agent": Agent("Analysis Agent", ANALYSIS_PROMPT),
        "Critic Agent": Agent("Critic Agent", CRITIC_PROMPT),
        "Response Agent": Agent("Response Agent", RESPONSE_PROMPT),
    }


In [ ]:
# CELL 7: Centralized Communication Framework
# A Coordinator controls every step. All information flows through it.
#
# Flow:
# User -> Coordinator -> Research Agent -> Coordinator -> Analysis Agent
#       -> Coordinator -> Critic Agent -> Coordinator -> Response Agent
#       -> Final Answer

def run_centralized(user_request, bus):
    agents = make_fresh_agents()
    coordinator = "Coordinator"

    # User sends the request to the Coordinator
    bus.send("User", coordinator, user_request)

    # Coordinator -> Research Agent
    bus.send(coordinator, "Research Agent", "Research task assigned.")
    research_output = agents["Research Agent"].run(user_request)
    bus.send("Research Agent", coordinator, "Research completed.")

    # Coordinator -> Analysis Agent
    bus.send(coordinator, "Analysis Agent", "Analyze the research findings.")
    analysis_input = f"User request: {user_request}\n\nResearch findings:\n{research_output}"
    analysis_output = agents["Analysis Agent"].run(analysis_input)
    bus.send("Analysis Agent", coordinator, "Analysis completed.")

    # Coordinator -> Critic Agent
    bus.send(coordinator, "Critic Agent", "Review the analysis for weaknesses.")
    critic_input = f"User request: {user_request}\n\nAnalysis:\n{analysis_output}"
    critic_output = agents["Critic Agent"].run(critic_input)
    bus.send("Critic Agent", coordinator, "Review completed.")

    # Coordinator -> Response Agent
    bus.send(coordinator, "Response Agent", "Produce the final answer.")
    response_input = (
        f"User request: {user_request}\n\n"
        f"Research:\n{research_output}\n\n"
        f"Analysis:\n{analysis_output}\n\n"
        f"Critique:\n{critic_output}"
    )
    final_output = agents["Response Agent"].run(response_input)
    bus.send("Response Agent", coordinator, "Final answer ready.")
    bus.send(coordinator, "User", "Here is the final answer.")

    flow_diagram = (
        "👤 User\n↓\n🎯 Coordinator\n↓\n🔎 Research Agent\n↓\n🎯 Coordinator\n↓\n"
        "📊 Analysis Agent\n↓\n🎯 Coordinator\n↓\n🧐 Critic Agent\n↓\n🎯 Coordinator\n↓\n"
        "✍️ Response Agent\n↓\n🎯 Coordinator\n↓\n👤 Final Answer"
    )

    return {
        "final_output": final_output,
        "flow_diagram": flow_diagram,
        "agents": agents,
    }


In [ ]:
# CELL 8: Decentralized Communication Framework
# No central Coordinator. Each agent, once finished, decides (via simple
# predefined routing) which agent to hand off to next.

# Predefined routing table: current agent -> next agent
DECENTRALIZED_ROUTE = {
    "Research Agent": "Analysis Agent",
    "Analysis Agent": "Critic Agent",
    "Critic Agent": "Response Agent",
}

def run_decentralized(user_request, bus):
    agents = make_fresh_agents()

    # The request starts directly with the Research Agent (no coordinator)
    current_agent_name = "Research Agent"
    context = user_request
    outputs = {}

    bus.send("User", current_agent_name, user_request)

    while True:
        agent = agents[current_agent_name]
        output = agent.run(context)
        outputs[current_agent_name] = output

        next_agent_name = DECENTRALIZED_ROUTE.get(current_agent_name)
        if next_agent_name is None:
            # Reached the end of the chain (Response Agent finished)
            break

        # This agent decides, using the predefined routing logic, to hand
        # off to the next agent directly (no coordinator involved).
        bus.send(current_agent_name, next_agent_name,
                  f"Passing along findings for further processing.")
        context = (
            f"User request: {user_request}\n\n"
            f"Previous agent ({current_agent_name}) output:\n{output}"
        )
        current_agent_name = next_agent_name

    final_output = outputs.get("Response Agent", "[No final output generated]")

    flow_diagram = (
        "🔎 Research Agent\n↓\n📊 Analysis Agent\n↓\n🧐 Critic Agent\n↓\n✍️ Response Agent"
    )

    return {
        "final_output": final_output,
        "flow_diagram": flow_diagram,
        "agents": agents,
    }


In [ ]:
# CELL 9: Parallel Communication Framework (asyncio)
# Research, Analysis and Critic agents all run CONCURRENTLY on the raw
# user request (simplified — no waiting on each other). Once all three
# finish, their outputs are combined and sent to the Response Agent.

import threading

async def _run_parallel_async(user_request, bus):
    agents = make_fresh_agents()

    bus.send("User", "Task Router", user_request)
    for name in ["Research Agent", "Analysis Agent", "Critic Agent"]:
        bus.send("Task Router", name, "Work on the user request concurrently.")

    # Run the three agents concurrently
    research_task = agents["Research Agent"].run_async(user_request)
    analysis_task = agents["Analysis Agent"].run_async(user_request)
    critic_task = agents["Critic Agent"].run_async(user_request)

    research_output, analysis_output, critic_output = await asyncio.gather(
        research_task, analysis_task, critic_task
    )

    for name, out in [
        ("Research Agent", research_output),
        ("Analysis Agent", analysis_output),
        ("Critic Agent", critic_output),
    ]:
        bus.send(name, "Task Router", "Finished processing.")

    bus.send("Task Router", "Response Agent", "Combine all outputs into a final answer.")
    combined_input = (
        f"User request: {user_request}\n\n"
        f"Research Agent output:\n{research_output}\n\n"
        f"Analysis Agent output:\n{analysis_output}\n\n"
        f"Critic Agent output:\n{critic_output}"
    )
    final_output = agents["Response Agent"].run(combined_input)
    bus.send("Response Agent", "Task Router", "Final answer ready.")

    return final_output, agents


def run_parallel(user_request, bus):
    """
    Synchronous wrapper that runs the async parallel framework and measures
    execution time.

    IMPORTANT: Colab/Jupyter (and Gradio itself, which runs a uvicorn server
    in a background thread) already have their own event loop running.
    Calling asyncio.run() directly on the main thread — or patching asyncio
    globally with nest_asyncio — can conflict with that and crash Gradio's
    server. The safe fix is to run our own small event loop inside a brand
    new, isolated background thread, which is guaranteed not to already have
    a loop running.
    """
    start_time = time.time()
    result_box = {}

    def _worker():
        result_box["value"] = asyncio.run(_run_parallel_async(user_request, bus))

    thread = threading.Thread(target=_worker)
    thread.start()
    thread.join()

    final_output, agents = result_box["value"]
    elapsed = time.time() - start_time

    flow_diagram = (
        "👤 User\n↓\n🔀 Task Router\n"
        "├── 🔎 Research Agent\n├── 📊 Analysis Agent\n└── 🧐 Critic Agent\n↓\n"
        "✍️ Response Agent\n↓\n👤 Final Answer"
    )

    return {
        "final_output": final_output,
        "flow_diagram": flow_diagram,
        "agents": agents,
        "elapsed": elapsed,
    }


In [ ]:
# CELL 10: Peer-to-Peer Communication Framework
# No central Coordinator. Agents communicate directly with each other
# through the MessageBus. The Critic can send work back to the Research
# Agent for ONE correction cycle at most, to avoid infinite loops.

def run_p2p(user_request, bus):
    agents = make_fresh_agents()
    correction_used = False

    # Research Agent starts, talking directly to Analysis Agent next
    research_output = agents["Research Agent"].run(user_request)
    bus.send("Research Agent", "Analysis Agent", "Here are the research findings.")

    analysis_input = f"User request: {user_request}\n\nResearch findings:\n{research_output}"
    analysis_output = agents["Analysis Agent"].run(analysis_input)
    bus.send("Analysis Agent", "Critic Agent", "Here is the analysis, please review.")

    critic_input = f"User request: {user_request}\n\nAnalysis:\n{analysis_output}"
    critic_output = agents["Critic Agent"].run(critic_input)

    # Simple rule: if the critic's text mentions correction-related keywords
    # AND we haven't already used our one allowed correction cycle, send the
    # work back to the Research Agent once.
    needs_correction = any(
        kw in critic_output.lower()
        for kw in ["missing", "incorrect", "error", "should include", "lacks", "correction"]
    )

    if needs_correction and not correction_used:
        correction_used = True
        bus.send("Critic Agent", "Research Agent",
                  "Correction requested — please refine the research with this feedback.")
        refine_input = (
            f"User request: {user_request}\n\n"
            f"Original research:\n{research_output}\n\n"
            f"Critic feedback:\n{critic_output}\n\n"
            f"Please refine/expand the research to address the feedback."
        )
        research_output = agents["Research Agent"].run(refine_input)
        bus.send("Research Agent", "Analysis Agent", "Here is the corrected research.")

        analysis_input = f"User request: {user_request}\n\nCorrected research:\n{research_output}"
        analysis_output = agents["Analysis Agent"].run(analysis_input)
        bus.send("Analysis Agent", "Critic Agent", "Here is the revised analysis.")

        critic_input = f"User request: {user_request}\n\nRevised analysis:\n{analysis_output}"
        critic_output = agents["Critic Agent"].run(critic_input)

    bus.send("Critic Agent", "Response Agent", "Review complete, please finalize the answer.")
    response_input = (
        f"User request: {user_request}\n\n"
        f"Research:\n{research_output}\n\n"
        f"Analysis:\n{analysis_output}\n\n"
        f"Critique:\n{critic_output}"
    )
    final_output = agents["Response Agent"].run(response_input)

    flow_diagram = (
        "🔎 Research Agent\n↓\n📊 Analysis Agent\n↓\n🧐 Critic Agent\n"
        + ("↓ (correction cycle used)\n🔎 Research Agent\n↓\n📊 Analysis Agent\n↓\n🧐 Critic Agent\n"
           if correction_used else "")
        + "↓\n✍️ Response Agent"
    )

    return {
        "final_output": final_output,
        "flow_diagram": flow_diagram,
        "agents": agents,
        "correction_used": correction_used,
    }


In [ ]:
# CELL 11: Common run_framework() function
# A single entry point used by the UI to run any of the four frameworks
# and collect all the metrics needed for display.

FRAMEWORKS = ["Centralized", "Decentralized", "Parallel", "Peer-to-Peer"]

def run_framework(framework_name, user_request):
    """
    Runs the selected framework on the given user request.
    Returns a dictionary with everything the UI needs:
    final answer, status per agent, flow diagram, communication log,
    and metrics (time, message count, LLM call count).
    """
    bus = MessageBus()
    llm_call_counter["count"] = 0   # reset real-API-call counter for this run
    demo_call_counter["count"] = 0  # reset demo-call counter for this run

    start_time = time.time()

    if framework_name == "Centralized":
        result = run_centralized(user_request, bus)
    elif framework_name == "Decentralized":
        result = run_decentralized(user_request, bus)
    elif framework_name == "Parallel":
        result = run_parallel(user_request, bus)
    elif framework_name == "Peer-to-Peer":
        result = run_p2p(user_request, bus)
    else:
        raise ValueError(f"Unknown framework: {framework_name}")

    elapsed = result.get("elapsed", time.time() - start_time)

    agent_status = {name: agent.status for name, agent in result["agents"].items()}

    # A run counts as DEMO MODE if any agent in it fell back to a simulated
    # response — even if it started on the real API and switched partway
    # through (e.g. credits ran out mid-run).
    execution_mode = "DEMO MODE" if demo_call_counter["count"] > 0 else "Gemini API"

    return {
        "framework": framework_name,
        "final_output": result["final_output"],
        "flow_diagram": result["flow_diagram"],
        "agent_status": agent_status,
        "communication_log": bus.get_log_text(),
        "execution_time": elapsed,
        "num_agents": len(result["agents"]),
        "num_messages": bus.count(),
        "num_llm_calls": llm_call_counter["count"],
        "num_demo_calls": demo_call_counter["count"],
        "execution_mode": execution_mode,
    }


In [ ]:
# CELL 12: Gradio UI

import gradio as gr
import pandas as pd

DEFAULT_REQUEST = (
    "Should a college develop an AI-powered attendance system? Analyze its "
    "benefits, challenges, cost considerations and technical feasibility."
)

STATUS_ORDER = ["Research Agent", "Analysis Agent", "Critic Agent", "Response Agent"]


def format_status_text(agent_status):
    lines = []
    for name in STATUS_ORDER:
        status = agent_status.get(name, "Waiting")
        emoji = {"Waiting": "⏳", "Running": "🔄", "Completed": "✅", "Failed": "❌"}.get(status, "⏳")
        lines.append(f"{emoji} {name}: {status}")
    return "\n".join(lines)


def run_single_framework_ui(user_request, framework_name):
    if not user_request or not user_request.strip():
        return ("Please enter a request.", "", "", "", "", "")

    # NOTE: we deliberately do NOT block here when client is None — the app
    # runs fine in DEMO MODE with no API key at all. call_llm() handles that.
    try:
        result = run_framework(framework_name, user_request)
    except Exception as e:
        # Keep this short — never show a full traceback to the user.
        error_msg = "⚠️ Something went wrong while running the framework. Please try again."
        print(f"[INFO] run_single_framework_ui error: {e}")
        return (error_msg, "", "", "", "", "")

    status_text = format_status_text(result["agent_status"])
    metrics_text = (
        f"Framework: {result['framework']}\n"
        f"Execution Time: {result['execution_time']:.2f} sec\n"
        f"Number of Agents: {result['num_agents']}\n"
        f"Number of Messages: {result['num_messages']}\n"
        f"Number of LLM Calls: {result['num_llm_calls']}\n"
        f"Execution Mode: {result['execution_mode']}"
    )

    return (
        result["final_output"],
        status_text,
        result["flow_diagram"],
        result["communication_log"],
        metrics_text,
        "",  # placeholder for comparison table (unused on single run)
    )


COMPARE_COLUMNS = ["Framework", "Execution Time (s)", "Messages", "LLM Calls", "Mode"]


def compare_all_frameworks_ui(user_request):
    if not user_request or not user_request.strip():
        empty_df = pd.DataFrame(columns=COMPARE_COLUMNS)
        return "Please enter a request.", empty_df

    # NOTE: no early "no API key" block here either — comparison runs fine
    # entirely in DEMO MODE if the API is unavailable.
    rows = []
    notes = []
    for fw in FRAMEWORKS:
        try:
            result = run_framework(fw, user_request)
            rows.append({
                "Framework": fw,
                "Execution Time (s)": round(result["execution_time"], 2),
                "Messages": result["num_messages"],
                "LLM Calls": result["num_llm_calls"],
                "Mode": result["execution_mode"],
            })
        except Exception as e:
            print(f"[INFO] compare_all_frameworks_ui error on {fw}: {e}")
            notes.append(f"{fw}: something went wrong, skipped.")
            rows.append({
                "Framework": fw,
                "Execution Time (s)": "N/A",
                "Messages": "N/A",
                "LLM Calls": "N/A",
                "Mode": "N/A",
            })

    df = pd.DataFrame(rows)
    summary = (
        "This is a simple experimental demonstration comparing the four "
        "communication frameworks on the same request — not a formal "
        "scientific benchmark."
    )
    if USE_DEMO_MODE:
        summary += (
            "\n\n⚠️ DEMO MODE — Gemini API unavailable"
            + (f" ({DEMO_MODE_REASON})" if DEMO_MODE_REASON else "")
            + ". Responses are simulated for demonstration."
        )
    if notes:
        summary += "\n\n" + "\n".join(notes)

    return summary, df


with gr.Blocks(title="Multi-Agent AI Communication Framework") as demo:
    gr.Markdown("# 🤖 Multi-Agent AI Communication Framework")
    gr.Markdown(
        "Compare how multiple AI agents communicate using Centralized, "
        "Decentralized, Parallel and Peer-to-Peer architectures."
    )

    with gr.Row():
        with gr.Column(scale=2):
            request_box = gr.Textbox(
                label="Enter your request",
                value=DEFAULT_REQUEST,
                lines=4,
            )
            framework_dropdown = gr.Dropdown(
                choices=FRAMEWORKS,
                value="Centralized",
                label="Communication Framework",
            )
            with gr.Row():
                run_button = gr.Button("🚀 Run Agents", variant="primary")
                compare_button = gr.Button("📊 Compare All Frameworks")

        with gr.Column(scale=1):
            gr.Markdown("### Agent Status")
            status_box = gr.Textbox(label="", lines=5, interactive=False)

    gr.Markdown("### Communication Flow")
    flow_box = gr.Textbox(label="", lines=8, interactive=False)

    gr.Markdown("### Communication Log")
    log_box = gr.Textbox(label="", lines=12, interactive=False)

    gr.Markdown("### Final Response")
    final_box = gr.Textbox(label="", lines=8, interactive=False)

    gr.Markdown("### Metrics")
    metrics_box = gr.Textbox(label="", lines=5, interactive=False)

    gr.Markdown("### Comparison: All Frameworks")
    compare_summary_box = gr.Textbox(label="", lines=3, interactive=False)
    compare_table = gr.Dataframe(
        headers=COMPARE_COLUMNS,
        interactive=False,
    )

    run_button.click(
        fn=run_single_framework_ui,
        inputs=[request_box, framework_dropdown],
        outputs=[final_box, status_box, flow_box, log_box, metrics_box, compare_summary_box],
    )

    compare_button.click(
        fn=compare_all_frameworks_ui,
        inputs=[request_box],
        outputs=[compare_summary_box, compare_table],
    )


In [ ]:
print("USE_DEMO_MODE =", USE_DEMO_MODE)
print("MODEL_NAME =", MODEL_NAME)
print("Client exists =", client is not None)

try:
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents="Reply with exactly: Gemini is working"
    )

    print("✅ Gemini is working")
    print(response.text)

except Exception as e:
    print("❌ Gemini test failed")
    print("Error type:", type(e).__name__)
    print("Error:", repr(e))

USE_DEMO_MODE = False
MODEL_NAME = gemini-flash-latest
Client exists = True
✅ Gemini is working
Gemini is working


In [ ]:
# CELL 13: Launch Gradio
# share=True gives you a public link that works well from Colab.
# debug=True streams print() output (like our DEMO MODE switch messages)
# live into this cell's output, which is handy while testing.
demo.launch( share=True, debug= True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://0bc7343ab12a797e56.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[INFO] Gemini API unavailable (Gemini service temporarily unavailable). Switching to DEMO MODE.


## How this project works

1. **Message class** — a small container holding `sender`, `receiver`, and `content`.
2. **MessageBus** — every time one agent (or the Coordinator/User/Task Router)
   sends something to another, it is recorded here. The full log is shown in the UI.
3. **Agent class** — a shared structure for all four agents. Each agent has a
   name, a system prompt (its role), and a `run()` method that calls the LLM.
4. **Four agents** — Research, Analysis, Critic, and Response — each with a
   distinct system prompt, as specified in the project brief.

## How each communication framework works

- **Centralized** — a `Coordinator` sits in the middle of every exchange. It
  hands the task to Research, then Analysis, then Critic, then Response, and
  relays results back and forth. Nothing goes directly agent-to-agent.
- **Decentralized** — there's no Coordinator. Each agent finishes its own work
  and looks up a small predefined routing table (`Research → Analysis →
  Critic → Response`) to decide who to hand off to next.
- **Parallel** — Research, Analysis, and Critic all run **at the same time**
  using `asyncio.gather`, each working directly from the original user
  request (for simplicity). This runs inside its own isolated background
  thread with a fresh event loop, so it never conflicts with Colab's or
  Gradio's own event loop. Once all three finish, their outputs are combined
  and sent to the Response Agent. The UI shows execution time so you can see
  the speed benefit versus running agents one after another.
- **Peer-to-Peer** — agents talk directly to each other through the
  MessageBus: Research → Analysis → Critic → Response. If the Critic's
  feedback suggests something is missing or incorrect, it sends the work
  back to the Research Agent for **one correction cycle only**, preventing
  infinite loops.

## How to run the notebook in Google Colab

1. Upload this notebook to Google Colab (or open it directly there).
2. Run **CELL 1** to install `gradio` and `google-genai`.
3. Run **CELL 2** and paste your Gemini API key when prompted (see below).
4. Run **CELLS 3–11** in order — these define the classes, agents, and the
   four frameworks. No output is expected from these; they just define code.
5. Run **CELL 12** to build the Gradio interface.
6. Run **CELL 13** to launch the app — a public Gradio link will appear.

## How to enter the API key

- Get a FREE key from Google AI Studio: https://aistudio.google.com/app/apikey
- When you run **CELL 2**, you'll be prompted with `Enter your Gemini API Key:`
  Paste it in and press Enter — it is never printed or saved to the notebook.
- Alternatively, set it beforehand as an environment variable or Colab secret
  named `GEMINI_API_KEY`, and CELL 2 will detect and reuse it automatically.

## DEMO MODE (no quota / no key required)

This project never depends on having Gemini API quota to demonstrate the
four communication architectures. `call_llm()` (CELL 5) automatically falls
back to a local, simulated response generator whenever the Gemini API is
unavailable — missing key, quota exceeded, invalid key, rate limit, or a
network error. The very first time this happens, a flag (`USE_DEMO_MODE`) is
set so the app stops retrying the failing API for the rest of the session,
and every subsequent response is clearly labeled:

> ⚠️ DEMO MODE — Gemini API unavailable. Response is simulated for demonstration.

Demo responses aren't one fixed sentence — they're built from the agent's
role, the actual request text, and the previous agent's output, so the
multi-agent flow still looks and feels real. The Metrics section always
shows an **Execution Mode** ("Gemini API" or "DEMO MODE") so it's obvious
which one was used, and the Compare table shows it per framework too.

## How to test the system

1. Keep the default example request or type your own.
2. Pick a framework from the dropdown (e.g. "Centralized") and click
   **🚀 Run Agents**. Watch the Agent Status, Communication Flow, and
   Communication Log update, and read the Final Response and Metrics.
3. Try switching the dropdown to each of the other three frameworks with the
   same request, and compare the communication logs and flow diagrams.
4. Click **📊 Compare All Frameworks** to run all four frameworks on the same
   request back-to-back and see a table of execution time, message count,
   and LLM call count for each.

## What to demonstrate during the project presentation

- Explain the shared `Agent` and `Message`/`MessageBus` structure and why a
  common structure makes it easy to plug in different frameworks.
- Run each framework once and point out **who talks to whom** in the
  Communication Log — this is the clearest way to show the structural
  difference between Centralized, Decentralized, Parallel, and P2P.
- Highlight the **Parallel** framework's execution time versus the others to
  show the concrete benefit of concurrent agent execution.
- Show the **Peer-to-Peer** correction cycle by picking a request likely to
  prompt critic feedback, and point out that the loop runs at most once.
- Use the **Compare All Frameworks** table as your closing slide, and be
  clear that it's an experimental demonstration on one sample request, not a
  rigorous scientific benchmark.
- Mention that the project degrades gracefully: if the API key is missing or
  out of credits, it automatically switches to a clearly-labeled DEMO MODE
  instead of failing, so the architecture demo never depends on live credits.
